# Rent model

This notebook builds a model that predicts the **yearly rent** of a house from the AQAR rental data. It follows `plan.md`:

- **Phase 1, explore (steps 1 to 8):** load the data, split off a test set, read the descriptions, add new columns and draw graphs.
- **Phase 2, clean (steps 9 and 10):** remove the rows that would teach the model wrong prices, then list the model's inputs.
- **Phase 3, models (steps 11 to 18):** grade every model the same way, beat two simple baselines, improve the best model, and take the final exam on the test set.

**Rules:** decide everything with `train`; `test` is used once, in step 18. Before calling a phase done, restart the kernel and click Run All. Before committing, click Clear All Outputs: the dataset's licence doesn't allow publishing its rows.

## Setup

The libraries this notebook uses:

- `pandas` for tables, `numpy` for maths and `matplotlib` for graphs
- `re` to search text with patterns, `zlib` for the test-set split and `os` to create a folder

The last line makes pandas show up to 200 characters of each description instead of 50.

In [ ]:
import os
import re
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 200)

A small helper that prints a dashed line between two outputs.

In [ ]:
def space():
    print("- -" * 50)

## Step 1: load the data and check it

Read the CSV file. The encoding `utf-8-sig` skips an invisible mark at the start of the file.

In [ ]:
df = pd.read_csv("data-raw/SA_Aqar.csv", encoding="utf-8-sig")

The first row, to see what one house looks like.

In [ ]:
df.head(1)

The number of rows and columns, the type of each column, and the column names.

In [ ]:
print(df.shape)
space()
print(df.dtypes)
space()
print(df.columns)

The same check as one table, with how many values each column has.

In [ ]:
df.info()

The smallest, largest and average value of each number column. Two sizes look wrong: 1 m² and 95,000 m². Step 4c reads their descriptions.

In [ ]:
df.describe()

Missing values in each column. Only `details`, the description, has some.

In [ ]:
df.isnull().sum()

## Step 2: drop the exact copies

Many ads appear more than once. `duplicated()` marks a row `True` when it repeats an earlier row exactly.

In [ ]:
print(df.duplicated())
df.tail(10)

The number of rows before anything is dropped.

In [ ]:
print(df.shape)

Which houses have copies, and how many of each. The description is left out to keep the table short.

In [ ]:
counts = df.value_counts(dropna=False).reset_index(name="copies")
groups = counts[counts["copies"] > 1]
print(len(groups), "listings have copies")
groups.drop(columns="details")

Keep the first row of each group of copies.

In [ ]:
df =df.drop_duplicates()

3,718 rows became 1,521.

In [ ]:
df.shape

## Step 3: split off a test set

The test set is a final exam: the model never sees these houses while it learns, so its score on them shows how it will do on houses it has never met.

### Why not a normal random split
The usual way is train_test_split(df, test_size=0.2, random_state=42) from scikit-learn. Its random_state also makes it repeatable, but it has two weaknesses here:

Copies can be split up. It decides row by row, so one copy of a house can land in train and the other in test. crc32 gives copies the same number, so they always land together.
Removing rows reshuffles everything. If you later drop rows, for example the near-copies or the 40 houses in the app's sample data, a seeded shuffle hides a different set of houses. With crc32, each house's place depends only on its own facts, so the houses that remain stay where they were.

### How the split works

- `FACTS` lists the 22 columns that describe a house: every column except `price` and `details`. They are written out by name, so new columns added later can't change the split.
- `house_text` joins one house's facts into one text.
- `is_hidden` turns that text into a number with `crc32`, and hides the house when the number divided by 5 leaves 0. That hides about 1 house in 5, and always the same ones.
- `split` returns `train` and `test`. It is called again whenever rows or columns change.

The first version of this cell kept the answer in a column, `is_test`, and built `FACTS` from `df.columns`. Running it twice counted `is_test` as a fact and hid different houses. This version gives the same answer however often it runs.

In [ ]:
FACTS = [
    "city", "district", "front", "size", "property_age", "bedrooms", "bathrooms",
    "livingrooms", "kitchen", "garage", "driver_room", "maid_room", "furnished", "ac",
    "roof", "pool", "frontyard", "basement", "duplex", "stairs", "elevator", "fireplace",
]


def house_text(row):
    """One house's facts joined into one text, with | between them."""
    return "|".join(str(value) for value in row)


def is_hidden(row):
    """True for about 1 house in 5: the houses kept for the test set."""
    number = zlib.crc32(house_text(row).encode())
    return number % 5 == 0


def split(data):
    """Return (train, test). The same house always lands on the same side."""
    hidden = data[FACTS].apply(is_hidden, axis=1)
    return data[~hidden], data[hidden]  # ~ means "not"


train, test = split(df)
print(len(train), "houses to learn from,", len(test), "kept for the final exam")

Check: 1,197 rows and the 24 original columns.

In [ ]:
train.shape

## Step 4: read the descriptions

The numbers can't say whether a price is monthly, or whether an ad rents a whole house. The descriptions can. Every decision in this step is made with `train`.

### Clean up the Arabic text first

The same word can be spelled several ways: شقة or شقه, أحمد or احمد. `normalise` writes every description in one spelling, so one search finds them all. It does exactly what `normalise()` in `frontend/src/utils/arabic.js` does:

1. lower-case the English letters
2. remove the short-vowel marks and the stretching line (ـ)
3. write أ إ آ ٱ as ا, ى as ي, and ة as ه
4. write the Arabic digits ٠ to ٩ as 0 to 9
5. leave one space between words

The result goes in a new column, `text`. It is only for searching: the model never sees it, and `details` stays as it was. Each `assert` line checks the function on an example and stops with an error if the answer is wrong.

In [ ]:
ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")


def normalise(text):
    """The text in one standard spelling, the same as normalise() in arabic.js."""
    text = text.lower()
    text = re.sub("[ً-ْـ]", "", text)  # short-vowel marks and the stretching line
    text = re.sub("[أإآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = text.translate(ARABIC_DIGITS)
    text = re.sub(r"\s+", " ", text)  # any run of spaces or line breaks becomes one space
    return text.strip()


assert normalise("الروضة") == "الروضه"
assert normalise("أحمد") == "احمد"

df["text"] = df["details"].fillna("").map(normalise)  # a missing description becomes ""
train, test = split(df)

cut_short = train["details"].fillna("").str.endswith("...")
print(cut_short.sum(), "of", len(train), "descriptions end in '...': the dataset kept only their beginning")

### Step 4a: are the prices yearly or monthly?

A monthly price is about 12 times smaller than a yearly one. If some ads give monthly prices, they show up as a separate group of very cheap houses.

`np.log10` counts the zeros: 10,000 becomes 4 and 100,000 becomes 5. On this scale, cheap and expensive houses fit on one chart.

In [ ]:
log_price = np.log10(train["price"])

The histogram. The red line marks 10,000 SAR.

In [ ]:
plt.hist(log_price, bins=50)
plt.axvline(4, color="red", linestyle="--")  # the red line marks 10,000 SAR
plt.xlabel("log10 of price (4 = 10,000 SAR, 5 = 100,000 SAR)")
plt.ylabel("number of houses")
plt.title("Step 4a: house prices on a log scale")
plt.show()

How many houses sit left of the red line.

In [ ]:
cheap = train["price"] < 10_000  # True for houses left of the red line
print(cheap.sum(), "houses cost under 10,000 SAR")

How often the descriptions say شهري (monthly) and سنوي (yearly). `pd.qcut` cuts the prices into 5 bands that each hold the same number of houses. The table gives the share of houses in each band whose description contains each word.

In [ ]:
bands = pd.qcut(train["price"], 5)
says_monthly = train["text"].str.contains("شهري")
says_yearly = train["text"].str.contains("سنوي")

word_share = pd.DataFrame({
    "houses": bands.value_counts(sort=False),
    "say monthly (%)": says_monthly.groupby(bands, observed=True).mean() * 100,
    "say yearly (%)": says_yearly.groupby(bands, observed=True).mean() * 100,
})
word_share.round(1)

The descriptions of the 20 cheapest houses.

In [ ]:
train.nsmallest(20, "price")[["price", "city", "details"]]

**What 4a found**

- Almost every price is yearly: the histogram has one big hump, from about 30,000 to 200,000 SAR.
- 21 houses cost under 10,000 SAR. Their descriptions show 6 monthly prices (شهري), 7 daily prices for holiday chalets in Jeddah (يومي, دره العروس), 1 house for sale, and 7 that give no clue but look monthly.
- One more house, at 14,000 SAR, gives a weekly price (اسبوعي).
- Above 10,000 SAR, شهري usually means "monthly payments accepted" or "the owner refuses monthly". The price itself is still yearly.

**Rule for step 9:** drop every price under 10,000 SAR, and every ad that mentions اسبوعي (weekly).

### Step 4b: which rows aren't houses?

For each ad, ask: **does it rent a whole house to live in?** An apartment, a single floor, a whole building, a holiday chalet or an office has different prices from a house, and would confuse the model.

| The ad says | Meaning | Whole house? |
| --- | --- | --- |
| فيلا … يوجد فيها شقه | a villa that has an apartment inside | ✔ |
| شقه في الدور الارضي مستقله | a separate ground-floor apartment | ✘ |
| فيلا للايجار دورين | a villa with two floors | ✔ |
| دور ارضي للايجار | a ground floor for rent | ✘ |
| فيلا مع استراحه | a villa with a rest house | ✔ |
| عماره جديده | a new building | ✘ |

First, three words anywhere in the ads. They are written with ه, not ة, because `normalise` changed it. `display` shows a table in the middle of a cell.

In [ ]:
for word in ["شقه", "عماره", "استراحه"]:
    has_word = train["text"].str.contains(word)
    print(word, ":", has_word.sum(), "houses")
    display(train.loc[has_word, ["price", "size", "details"]])

Most of those are villas that **have** an apartment or a rest house, so these words can't decide on their own. A better clue is how the ad starts: the first word usually says what is rented (فيلا, دبلكس, دور, شقه…).

`first_word` skips what comes before that word: symbols and emojis, a greeting (بسم الله, السلام عليكم) and the words for "for rent" (للايجار and its spellings). `\b` in the pattern means "end of a word", so ايجار is skipped but ايجارها (its rent) isn't.

⚠️ Searching for دور anywhere doesn't work: it's in 639 of the 1,197 ads, because it's also part of دورين (two floors) and دورات مياه (bathrooms). As the first word, it means "a floor".

In [ ]:
NOT_ARABIC = r"^[^ء-ي]+"  # anything at the start that isn't an Arabic letter
OPENING = r"^(بسم الله الرحمن الرحيم|السلام عليكم( ورحمه الله)?( وبركاته)?|للايجار|للاجار|للا يجار|لايجار|اللايجار|ايجار|يوجد لدينا|يوجد)\b"


def first_word(text):
    """The first real word of an ad, which usually says what it rents."""
    for _ in range(3):  # an ad can open with a greeting and then "for rent"
        text = re.sub(NOT_ARABIC, "", text)
        text = re.sub(OPENING, "", text)
    text = re.sub(NOT_ARABIC, "", text)
    words = text.split()
    return words[0] if words else ""


assert first_word("✨للايجار ✨ دور ارضي") == "دور"
assert first_word("السلام عليكم ورحمه الله للايجار فيلا") == "فيلا"

train["text"].map(first_word).value_counts().head(20)

The blank line in that list counts the ads with no description, or with no Arabic word in it.

Now the ads whose first word names something other than a house: دور (a floor), شقه or شقق (apartments), عماره (a building), شاليه (a chalet) or استراحه (a rest house).

مبنى (a building) is left out on purpose: `normalise` writes it as مبني, which also means "built", as in مبني بمواصفات ارامكو (built to Aramco standards), a house.

In [ ]:
NOT_A_HOUSE = ["دور", "شقه", "شقق", "عماره", "شاليه", "استراحه"]

starts_with = train["text"].map(first_word)
not_a_house = starts_with.isin(NOT_A_HOUSE)
print(not_a_house.sum(), "ads start with a word for something that isn't a house")
train.loc[not_a_house, ["price", "size", "details"]]

Three more kinds of ad that aren't a home to live in:

- **hotel villas** (فندقيه), rented by the month with hotel services
- **offices** (مكتبيه, اداري): villas offered as offices or company headquarters, and an office building
- **sales** (للبيع) that never mention rent. "For sale" is searched as للبيع, because بيع on its own hides inside other words: الربيع (spring) and طبيعي (natural).

In [ ]:
RENT_WORDS = "ايجار|اجار|تاجير|في السنه|سنوي"

hotel = train["text"].str.contains("فندقيه")
office = train["text"].str.contains("مكتبيه|اداري")
for_sale = train["text"].str.contains("للبيع") & ~train["text"].str.contains(RENT_WORDS)

for name, rows in [("hotel", hotel), ("office", office), ("for sale, no rent", for_sale)]:
    print(name, ":", rows.sum(), "houses")
    display(train.loc[rows, ["price", "size", "details"]])

**What 4b found**

- شقه, عماره and استراحه inside an ad mostly describe villas that include an apartment or a rest house. Those stay.
- 29 ads start with a word for something that isn't a house: mostly single floors (دور ارضي, دور اول), plus a few apartments, buildings and chalets. Three of them are whole villas and are lost too: two are described floor by floor ("دور ارضي … الدور الثاني …"), and one says دور واحد, meaning a single-storey villa. That is a small price for a rule that is simple and works the same way on the test set.
- 4 hotel-villa ads, 9 office ads and 4 sales that never mention rent. A few of them are also caught by the price rules.
- In all, 42 of the 1,197 ads in train (3.5%) are not a house to live in. So the SRS assumption A-06, "every listing is a villa or a duplex", holds for about 96% of the ads.

**Rules for step 9:** drop the ads whose first word is in `NOT_A_HOUSE`, the hotel and office ads, and the sales that never mention rent.

### Step 4c: impossible sizes and prices

The smallest and largest sizes, with their descriptions.

In [ ]:
print(train["size"].describe())
display(train.nsmallest(10, "size")[["price", "size", "city", "details"]])
train.nlargest(15, "size")[["price", "size", "city", "details"]]

The 20 most expensive houses.

In [ ]:
train.nlargest(20, "price")[["price", "size", "city", "details"]]

**What 4c found**

- One size is impossible: 1 m². The next smallest, 100 to 140 m², are real small houses and compound units.
- All 8 ads of 2,000 m² or more are not one house: 5 compounds of 5 to 108 villas, one villa whose 2,500 m² can't be its own size, a commercial property, and a villa whose description says 250 m² but was typed as 95,000. From 1,500 to 2,000 m² the ads are large villas.
- Every price above 500,000 SAR is wrong for a yearly rent: 4 sale prices (للبيع, السوم 800 الف, a mortgage ad, a 300 m² villa at 1.3 million), a whole compound of 11 villas, an office building, a commercial property, and two 300 m² villas at 550,000 that probably have an extra zero. At 450,000 SAR and below they are mostly real luxury and compound villas (one at 450,000 says 45000 in its description).

**Rules for step 9:** drop sizes under 50 m² or of 2,000 m² and more, and prices over 500,000 SAR.

### Step 4d: keyword candidates

Some things that change the price aren't columns, but the descriptions mention them. For each candidate word: how many houses mention it, and the median price with and without it. Riyadh only, so the comparison isn't just Riyadh against the cheaper cities.

`(?! به)` after ملحق means "not followed by به": ملحق به means "attached to it", for example a bathroom attached to a bedroom, not an annex.

In [ ]:
riyadh = train[train["city"].str.strip() == "الرياض"]
CANDIDATES = {
    "annex (ملحق)": r"ملحق(?! به)",
    "internal staircase (درج داخلي)": "درج داخلي",
    "apartment inside (شقه)": "شقه",
    "new (جديد)": "جديد",
    "luxury (فاخر)": "فاخر",
}

rows = []
for name, pattern in CANDIDATES.items():
    has = riyadh["text"].str.contains(pattern)
    rows.append({
        "keyword": name,
        "houses": has.sum(),
        "share (%)": round(has.mean() * 100, 1),
        "median with": riyadh.loc[has, "price"].median(),
        "median without": riyadh.loc[~has, "price"].median(),
    })
keywords = pd.DataFrame(rows)
keywords["with / without"] = (keywords["median with"] / keywords["median without"]).round(2)
keywords

**What 4d found**

- **ملحق (annex):** 14.7% of Riyadh houses mention one, and their median rent is 1.20 times that of the others (90,000 against 75,000 SAR). A clear signal.
- **درج داخلي (internal staircase):** 16.3% of the houses, with a median 0.59 times that of the others (50,000 against 85,000 SAR). These are the smaller villas built close together.
- **شقه** makes almost no difference. **جديد** and **فاخر** do, but the age column already covers new houses, and فاخر is in too few houses to learn from.

**For step 5:** two keyword flags, `has_annex` and `has_internal_stairs`.

## Step 5: feature engineering

New columns built from the old ones. Every new value goes in a **new** column: the 22 FACTS columns are never changed, so `split` keeps giving the same answer.

### City, front and type in English

The English names come from SRS Appendix A.3, so the model's values match the app's. `map` replaces each Arabic value with its English name, and `np.where` picks Duplex where `duplex` is 1 and Villa elsewhere. The `assert` lines stop with an error if a value is missing from the tables.

In [ ]:
CITY_NAMES = {"الرياض": "Riyadh", "جدة": "Jeddah", "الدمام": "Dammam", "الخبر": "Al Khobar"}
FRONT_NAMES = {
    "شمال": "North", "جنوب": "South", "شرق": "East", "غرب": "West",
    "شمال شرقي": "Northeast", "شمال غربي": "Northwest",
    "جنوب شرقي": "Southeast", "جنوب غربي": "Southwest",
    "3 شوارع": "Three streets", "4 شوارع": "Four streets",
}

df["city_en"] = df["city"].str.strip().map(CITY_NAMES)
df["front_en"] = df["front"].str.strip().map(FRONT_NAMES)
df["type"] = np.where(df["duplex"] == 1, "Duplex", "Villa")

assert df["city_en"].notna().all(), "a city is missing from CITY_NAMES"
assert df["front_en"].notna().all(), "a front is missing from FRONT_NAMES"

for column in ["city_en", "front_en", "type"]:
    print(df[column].value_counts(), "\n")

### District

The same district is written in several ways: "حي العارض", " العارض ", "العارضة". `district_name` applies `normalise` and removes a leading "حي " (district), as the app does when it compares districts. In step 14 the model learns each district's typical rent from these names.

In [ ]:
def district_name(district):
    """A district's name in one spelling, without the word حي in front."""
    name = normalise(district)
    return re.sub(r"^حي\s+", "", name)


assert district_name("   حي العارض ") == "العارض"
assert district_name("حي الروضة") == "الروضه"

df["district_norm"] = df["district"].map(district_name)
print(df["district"].nunique(), "district spellings became", df["district_norm"].nunique(), "districts")

### New numbers

| Column | What it is | Why it may help |
| --- | --- | --- |
| `log_size` | log10 of the size | Rent grows more slowly than size: 1,000 m² doesn't rent for twice as much as 500 m² |
| `total_rooms` | bedrooms + bathrooms + living rooms | One number for how big the house is inside |
| `size_per_bedroom` | size divided by bedrooms | A few big rooms or many small ones |
| `is_new` | 1 when the age is 0 | Brand-new houses may rent for more |
| `amenity_count` | how many of the 13 amenities it has | One number for how well equipped it is |
| `is_corner` | 1 when it faces 3 or 4 streets | Corner plots are often wanted |

`clip(lower=1)` treats 0 bedrooms as 1, so the division never divides by zero. `astype(int)` turns True and False into 1 and 0.

In [ ]:
AMENITIES = [
    "kitchen", "garage", "driver_room", "maid_room", "furnished", "ac", "roof",
    "pool", "frontyard", "basement", "stairs", "elevator", "fireplace",
]

df["log_size"] = np.log10(df["size"])
df["total_rooms"] = df["bedrooms"] + df["bathrooms"] + df["livingrooms"]
df["size_per_bedroom"] = df["size"] / df["bedrooms"].clip(lower=1)
df["is_new"] = (df["property_age"] == 0).astype(int)
df["amenity_count"] = df[AMENITIES].sum(axis=1)
df["is_corner"] = df["front_en"].isin(["Three streets", "Four streets"]).astype(int)

df[["log_size", "total_rooms", "size_per_bedroom", "is_new", "amenity_count", "is_corner"]].describe().round(2)

### Keyword flags from step 4d

1 when the description mentions it, 0 when it doesn't. A house with no description gets 0.

In [ ]:
df["has_annex"] = df["text"].str.contains(r"ملحق(?! به)").astype(int)
df["has_internal_stairs"] = df["text"].str.contains("درج داخلي").astype(int)

print("share of houses with each flag:")
print(df[["has_annex", "has_internal_stairs"]].mean().round(3))

Make `train` and `test` again, so they have the new columns.

In [ ]:
train, test = split(df)
print("train:", train.shape, " test:", test.shape)

## Step 6: delete the true duplicates

Step 2 removed rows that were the same in every column. Some houses were also posted twice with a **different description**: same facts, same price, same keyword flags. That is still one house, so only the first row stays.

In [ ]:
SAME_HOUSE = FACTS + ["has_annex", "has_internal_stairs", "price"]

print("before:", len(df), "rows")
df = df.drop_duplicates(subset=SAME_HOUSE)
print("after: ", len(df), "rows")

train, test = split(df)
print(len(train), "in train,", len(test), "in test")

## Step 7: graphs

All the graphs use `train`. The labels are in English, because matplotlib draws Arabic letters unjoined and backwards. Each graph is also saved in `figures/`, which git ignores, for the write-up later. The histogram of log price is in step 4a.

### Price against size, by city

One dot per house, one colour per city. Dots that rise to the right mean bigger houses rent for more.

In [ ]:
os.makedirs("figures", exist_ok=True)

for city, houses in train.groupby("city_en"):
    plt.scatter(houses["log_size"], np.log10(houses["price"]), s=8, alpha=0.5, label=city)
plt.xlabel("log10 of size (2.5 = 316 m², 3 = 1,000 m²)")
plt.ylabel("log10 of price (5 = 100,000 SAR)")
plt.title("Price against size, by city")
plt.legend()
plt.savefig("figures/price-vs-size.png", dpi=120, bbox_inches="tight")
plt.show()

### Median price by city, by type and by bedrooms

The median is the middle price: half the houses cost less and half cost more. Unlike the average, one very expensive house can't pull it up. Only about 20 houses have 1 or 2 bedrooms, so those two bars are less certain.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train.groupby("city_en")["price"].median().sort_values().plot.bar(ax=axes[0])
axes[0].set_title("Median price by city")

train.groupby("type")["price"].median().plot.bar(ax=axes[1])
axes[1].set_title("Median price by type")

train.groupby("bedrooms")["price"].median().plot.bar(ax=axes[2])
axes[2].set_title("Median price by bedrooms")

for ax in axes:
    ax.set_ylabel("SAR per year")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig("figures/median-price.png", dpi=120, bbox_inches="tight")
plt.show()

### Median price with and without each amenity, Riyadh only

A bar at 1.5 means houses with that amenity have a median price 1.5 times that of houses without it. Below 1, the houses with it are cheaper. That doesn't mean the amenity lowers the rent: cheaper areas may simply have more of it.

In [ ]:
riyadh = train[train["city_en"] == "Riyadh"]

ratios = {}
for column in AMENITIES + ["has_annex", "has_internal_stairs"]:
    has = riyadh[column] == 1
    ratios[column] = riyadh.loc[has, "price"].median() / riyadh.loc[~has, "price"].median()

pd.Series(ratios).sort_values().plot.barh(figsize=(7, 6))
plt.axvline(1, color="red", linestyle="--")  # 1 means no difference
plt.xlabel("median price with ÷ median price without")
plt.title("Riyadh: price with and without each amenity")
plt.savefig("figures/amenities.png", dpi=120, bbox_inches="tight")
plt.show()

### Houses per district

matplotlib can't draw the Arabic names, so this one is a table. Districts with few houses matter for step 14: the model can learn little about a district it has seen only once or twice.

In [ ]:
per_district = train["district_norm"].value_counts()
print(len(per_district), "districts in train,", (per_district < 5).sum(), "of them with fewer than 5 houses")
per_district.head(15)

### Do size and the room counts move together?

Spearman correlation compares the order of the values: 1 means that when one goes up, the other always goes up too; 0 means no link. When two columns are strongly linked, a model gets little extra from the second one.

In [ ]:
train[["size", "bedrooms", "bathrooms", "livingrooms", "price"]].corr(method="spearman").round(2)

## Step 8: checkpoint

### What the graphs show

- **Size matters, but less than you might expect.** Price rises with size, yet houses of the same size can differ by a factor of 3 or more (Spearman 0.30 between size and price). Where a house is, and how it is equipped, matter as much. The dots far below the cloud (under 10,000 SAR) and the two at sizes 1 and 95,000 are the problems step 9 removes.
- **City:** Jeddah has the highest median rent (100,000 SAR), then Riyadh (about 77,500), Al Khobar (65,000) and Dammam (60,000).
- **Type:** villas and duplexes are close, 80,000 against 75,000.
- **Bedrooms:** more bedrooms doesn't mean more rent. The medians stay between 60,000 and 85,000, and Spearman between bedrooms and price is −0.02. Bedrooms stop at 7 in the data, and big old houses with many rooms are often cheap.
- **Amenities (Riyadh):** a basement, air conditioning, an elevator, a pool, a driver's room, furniture and a maid's room each go with a median price 1.7 to 2 times higher. A fireplace and an internal staircase go with lower prices, most likely because they are common in the cheaper districts.
- **Size and the room counts** are only weakly linked (0.12 to 0.24), so each adds information and all of them stay.
- **Districts:** 164 in train, 99 of them with fewer than 5 houses. The target encoder in step 14 must not trust a district it has seen only a few times. scikit-learn's `TargetEncoder` handles that by blending each district's average with the overall average.

### The cleaning rules

Each rule removes rows that would teach the model a wrong price. None removes a row just because it is unusual or hard to predict.

| Rule | Removes | Found in |
| --- | --- | --- |
| 1. Prices that aren't yearly | prices under 10,000 SAR (monthly, daily, a sale) and weekly prices (اسبوعي) | 4a |
| 2. Not a whole house | ads whose first word is a floor, an apartment, a building, a chalet or a rest house | 4b |
| 3. Not a home | hotel villas (فندقيه) and offices (مكتبيه, اداري) | 4b |
| 4. For sale | ads that say للبيع and never mention rent | 4b |
| 5. Impossible sizes | under 50 m², or 2,000 m² and more | 4c |
| 6. Impossible prices | over 500,000 SAR | 4c |

Seen while reading, and kept:

- Some ads give two prices: pay at once or in two payments, furnished or not. The price column holds one of them, and the gap is 5 to 25%. That is noise a model can live with.
- Two ads rent two villas for one price, and one house was typed as 450,000 while its description says 45000. They are too few for a rule; the error analysis in step 17 will show whether they matter.
- About 40% of the descriptions are cut short, so a few ads may hide a monthly price or a sale in the missing part.

## Step 9: cleaning rules

Each rule is applied to `df`, so the test set is cleaned in exactly the same way as train. The rules run in order, so a row removed by an earlier rule isn't counted again.

`drop_where` does the work for every rule: it takes a column of True and False (True means remove), prints how many rows leave train and test, and returns `df` without them.

In [ ]:
def drop_where(data, remove, reason):
    """Remove the rows where `remove` is True, and print how many leave train and test."""
    hidden = data[FACTS].apply(is_hidden, axis=1)
    from_train = (remove & ~hidden).sum()
    from_test = (remove & hidden).sum()
    print(f"{reason}: {from_train} rows from train, {from_test} from test")
    return data[~remove]

**Rule 1: prices that aren't yearly** (step 4a).

In [ ]:
not_yearly = (df["price"] < 10_000) | df["text"].str.contains("اسبوعي")
df = drop_where(df, not_yearly, "Rule 1, prices that aren't yearly")

**Rule 2: not a whole house** (step 4b).

In [ ]:
not_a_house = df["text"].map(first_word).isin(NOT_A_HOUSE)
df = drop_where(df, not_a_house, "Rule 2, not a whole house")

**Rule 3: not a home** (step 4b): hotel villas and offices.

In [ ]:
not_a_home = df["text"].str.contains("فندقيه|مكتبيه|اداري")
df = drop_where(df, not_a_home, "Rule 3, hotel villas and offices")

**Rule 4: for sale** (step 4b): ads that say للبيع and never mention rent.

In [ ]:
for_sale = df["text"].str.contains("للبيع") & ~df["text"].str.contains(RENT_WORDS)
df = drop_where(df, for_sale, "Rule 4, for sale")

**Rule 5: impossible sizes** (step 4c).

In [ ]:
bad_size = (df["size"] < 50) | (df["size"] >= 2_000)
df = drop_where(df, bad_size, "Rule 5, impossible sizes")

**Rule 6: impossible prices** (step 4c).

In [ ]:
bad_price = df["price"] > 500_000
df = drop_where(df, bad_price, "Rule 6, impossible prices")

### The clean data

Make `train` and `test` again from the clean `df`. Every house that stayed is on the same side of the split as before.

In [ ]:
train, test = split(df)
print(len(df), "houses:", len(train), "in train and", len(test), "in test")
space()
print(train["city_en"].value_counts())
space()
print(train["price"].describe().round(0))

The histogram from step 4a again, now on the clean `train`: one hump, and nothing left of 10,000 SAR.

In [ ]:
plt.hist(np.log10(train["price"]), bins=50)
plt.axvline(4, color="red", linestyle="--")  # 10,000 SAR
plt.xlabel("log10 of price (4 = 10,000 SAR, 5 = 100,000 SAR)")
plt.ylabel("number of houses")
plt.title("Clean train: house prices on a log scale")
plt.savefig("figures/clean-price.png", dpi=120, bbox_inches="tight")
plt.show()

## Step 10: the model's inputs and its answer

`FEATURES` lists the columns the model may use. They come in two groups, because step 14 prepares them differently:

- `CATEGORICAL`: names, not amounts. Step 14 turns them into numbers: one-hot for city, front and type, and a target encoder for district.
- `NUMERIC`: amounts and 0/1 flags.

Not inputs: `price`, which is the answer the model learns to give (`TARGET`); `details` and `text`, which the keyword flags already summarise; the Arabic `city`, `district` and `front`, replaced by their cleaned versions; `size`, replaced by `log_size`; and `duplex`, replaced by `type`.

In [ ]:
CATEGORICAL = ["city_en", "district_norm", "front_en", "type"]
NUMERIC = [
    "log_size", "size_per_bedroom", "property_age", "is_new",
    "bedrooms", "bathrooms", "livingrooms", "total_rooms",
    "amenity_count", "is_corner", "has_annex", "has_internal_stairs",
] + AMENITIES
FEATURES = CATEGORICAL + NUMERIC
TARGET = "price"

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(len(FEATURES), "inputs:", len(CATEGORICAL), "categorical and", len(NUMERIC), "numeric")
print("train:", X_train.shape, " test:", X_test.shape)
print("missing values in the inputs:", X_train.isna().sum().sum() + X_test.isna().sum().sum())

**The data is ready.** The models learn from `X_train` and `y_train` in steps 12 to 17. `X_test` and `y_test` wait for the final exam in step 18.

| Stage | Rows |
| --- | --- |
| The raw CSV | 3,718 |
| After dropping the exact copies (step 2) | 1,521 |
| After dropping the true duplicates (step 6) | 1,513 |
| After the cleaning rules (step 9) | 1,418: 1,114 in train and 304 in test |

Next: Phase 3 in `plan.md`, starting with step 11, installing scikit-learn.

# Phase 3: the models

## Step 11: scikit-learn

scikit-learn is the library with the models and the tools to test them. It is already installed in this notebook's Python, and `requirements.txt` pins the same version, `scikit-learn==1.7.2`, for the rest of the team.

In [ ]:
import sklearn

print("scikit-learn", sklearn.__version__)

## Step 12: how every model is graded

**Cross-validation** works like practice exams: cut `train` into 5 parts, teach the model on 4 of them and let it guess the 5th, and repeat until every part has been guessed. At the end, every house has one guess from a model that never saw it.

- `groups`: 8 houses in train are the same house listed at two prices. `GroupKFold` keeps each pair in the same part, so a model is never graded on a house it has already seen.
- `three_numbers` grades a set of guesses. For a house that rents for 100,000 SAR and is guessed at 90,000, the miss is 10,000 SAR (it counts in **MAE**), that is 10% of the rent (it counts in **MAPE**), and 10% is under 15% (the house counts as **within 15%**).
- `score` makes the cross-validated guesses and grades them. **The lowest MAPE decides the winner.**

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict

groups = train[FACTS].apply(house_text, axis=1)  # copies of a house get the same text
folds = GroupKFold(n_splits=5)                    # 5 parts; one group never sits in two parts


def three_numbers(truth, guesses):
    """How good a set of guesses is: the average miss in % and in SAR, and the share within 15%."""
    miss = (guesses - truth).abs()  # how far each guess is from the real rent, in SAR
    share = miss / truth            # the same miss as a share of the rent: 0.10 = 10%
    return {
        "MAPE (%)": round(share.mean() * 100, 1),
        "MAE (SAR)": round(miss.mean()),
        "within 15% (%)": round((share <= 0.15).mean() * 100, 1),
    }


def score(model, columns):
    """Cross-validate `model` on train, using only `columns`, and return its three numbers."""
    guesses = cross_val_predict(model, X_train[columns], y_train, cv=folds, groups=groups)
    return three_numbers(y_train, guesses)

## Step 13: the baselines

A baseline is the simplest possible guess. If a clever model can't beat "the usual rent in that city", it is useless.

- `OneHotEncoder` turns the city into 4 yes/no columns, because a model can only do maths with numbers. Jeddah becomes Al Khobar 0, Dammam 0, Jeddah 1, Riyadh 0.
- `LinearRegression` learns one weight per column and adds them up. With only the city columns, it learns each city's usual rent.
- `make_pipeline` chains the steps: first the city becomes numbers, then the model learns. In every round of the cross-validation, the whole chain learns from the 4 training parts only.
- `on_log_scale` makes the model learn log(price). A 10,000 SAR miss is big on a 40,000 house and small on a 400,000 one; on the log scale the model cares about the miss in %. `np.exp` turns its guesses back into SAR.

**B1** knows only the city.

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder


def on_log_scale(model):
    """Train `model` on log(price), then turn its guesses back into SAR."""
    return TransformedTargetRegressor(regressor=model, func=np.log, inverse_func=np.exp)


results = {}

b1 = on_log_scale(make_pipeline(OneHotEncoder(), LinearRegression()))
results["B1: city only"] = score(b1, ["city_en"])

**B2** knows the city and the size. `ColumnTransformer` prepares each column its own way: one-hot for the city, while `log_size` passes through unchanged.

In [ ]:
city_one_hot = ColumnTransformer(
    [("city", OneHotEncoder(), ["city_en"])],
    remainder="passthrough",  # the other column, log_size, goes through unchanged
)
b2 = on_log_scale(make_pipeline(city_one_hot, LinearRegression()))
results["B2: city + size"] = score(b2, ["city_en", "log_size"])

pd.DataFrame(results).T

Knowing only the city, the guess is off by 44% on average, and only about 1 house in 5 is within 15%. Adding the size helps only a little. That matches step 7: houses of the same size can differ by 3 times in rent. The score to beat is B2's 41.4%.

## Step 14: Ridge regression

The same idea as `LinearRegression`, but with all 29 inputs. Some inputs say almost the same thing: `total_rooms` is the sum of the three room counts. Plain linear regression can then give silly weights, such as +1,000 to one input and −1,000 to its twin. **Ridge keeps every weight small**, and `alpha` sets how strongly.

First, the inputs become numbers:

- `OneHotEncoder` for city, front and type. `handle_unknown="ignore"` matters because a rare front, such as "Four streets", may be missing from the 4 training parts. Without it, the model would stop with an error.
- `TargetEncoder` for the district. One-hot would make 164 columns, most of them for districts with 1 to 4 houses. Instead, it replaces each district with its typical rent, learned from the training parts. A district seen only a few times leans towards the overall average, and a house's own rent is never used for that same house. `random_state=0` keeps its inner shuffle the same on every run.
- `StandardScaler` puts every number on the same scale, so `size_per_bedroom` (about 80) and `is_new` (0 or 1) are treated fairly when Ridge keeps the weights small.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, TargetEncoder

NAMES = ["city_en", "front_en", "type"]  # names with only a few possible values

ridge_inputs = ColumnTransformer([
    ("names", OneHotEncoder(handle_unknown="ignore"), NAMES),
    ("district", TargetEncoder(random_state=0), ["district_norm"]),
    ("numbers", StandardScaler(), NUMERIC),
])

Four strengths, from weak (0.1) to strong (100).

In [ ]:
tries = {}
for alpha in [0.1, 1, 10, 100]:
    ridge = on_log_scale(make_pipeline(ridge_inputs, Ridge(alpha=alpha)))
    tries[f"alpha = {alpha}"] = score(ridge, FEATURES)

pd.DataFrame(tries).T

Strengths 0.1 to 10 are equally good, and 100 is too strong. Keep 1.

In [ ]:
ridge = on_log_scale(make_pipeline(ridge_inputs, Ridge(alpha=1)))
results["Ridge"] = score(ridge, FEATURES)

## Step 15: gradient boosting

Many small decision trees. Each tree is a set of yes/no questions, such as "Is it in Jeddah? Is it over 400 m²?", and each new tree corrects part of the mistakes of the trees before it. Unlike a weighted sum, trees can learn combinations. Trees ask "is it bigger than X?", so the numbers don't need scaling.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

tree_inputs = ColumnTransformer(
    [
        ("names", OneHotEncoder(handle_unknown="ignore"), NAMES),
        ("district", TargetEncoder(random_state=0), ["district_norm"]),
    ],
    remainder="passthrough",  # the numbers go through unchanged
)

Eight settings. This cell takes about 20 seconds.

- `learning_rate`: how much each tree corrects. Smaller is more careful.
- `max_leaf_nodes`: how many different answers one tree can give. Smaller means simpler trees.
- `max_iter`: how many trees.

In [ ]:
tries = {}
for learning_rate in [0.05, 0.1]:
    for max_leaf_nodes in [15, 31]:
        for max_iter in [200, 500]:
            boosting = on_log_scale(make_pipeline(
                tree_inputs,
                HistGradientBoostingRegressor(
                    learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes, max_iter=max_iter,
                ),
            ))
            tries[f"{learning_rate} / {max_leaf_nodes} / {max_iter}"] = score(boosting, FEATURES)

pd.DataFrame(tries).T.sort_values("MAPE (%)")

The most careful setting wins: 0.05 / 15 / 200. More trees made it **worse**: with only 1,114 houses, extra trees start memorising the training houses instead of learning general rules. Keep the best.

In [ ]:
boosting = on_log_scale(make_pipeline(
    tree_inputs,
    HistGradientBoostingRegressor(learning_rate=0.05, max_leaf_nodes=15, max_iter=200),
))
results["Boosting"] = score(boosting, FEATURES)

## Step 16: the comparison

In [ ]:
pd.DataFrame(results).T.sort_values("MAPE (%)")

Both models almost halve the baselines' error, from 41% to about 23%. Boosting is ahead of Ridge by only 0.5 points, so by the rule in `plan.md` (within about 1 point, choose Ridge) Ridge would win. Step 17 looks for ways to miss less.

## Step 17: where the model misses, and how to miss less

### Error analysis

`cross_val_predict` gives Ridge's guess for every house in train, each from a model that never saw that house. `miss` is each guess's error as a share of the real rent, so 0.25 means 25%.

In [ ]:
guesses = pd.Series(
    cross_val_predict(ridge, X_train, y_train, cv=folds, groups=groups), index=y_train.index
)
miss = (guesses - y_train).abs() / y_train

print("MAPE by city (%):")
print((miss.groupby(train["city_en"]).mean() * 100).round(1))
space()
print("MAPE by price band (%):")
print((miss.groupby(pd.qcut(y_train, 5), observed=True).mean() * 100).round(1))

How much of the error comes from the worst houses, and the 10 biggest misses. Read the descriptions on screen only: they can hold phone numbers.

In [ ]:
worst = miss.sort_values(ascending=False)
worst_5_percent = worst.head(len(worst) // 20)
print(f"the worst 5% of houses make {worst_5_percent.sum() / worst.sum():.0%} of the total error")

biggest = train.loc[worst.index[:10], ["price", "city_en", "size", "details"]].copy()
biggest.insert(1, "guess", guesses[worst.index[:10]].round(-3))
biggest

Does the same district name exist in more than one city?

In [ ]:
cities_per_name = train.groupby("district_norm")["city_en"].nunique()
shared_names = cities_per_name[cities_per_name > 1].index
print(len(shared_names), "district names exist in more than one city, used by",
      train["district_norm"].isin(shared_names).sum(), "houses")

### What the analysis found

- **By city:** Al Khobar (29.6%) and Jeddah (26.5%) are the hardest, and Dammam the easiest (18.0%).
- **By price:** the cheapest fifth (31.8%) and the most expensive fifth (27.1%) miss most. The guesses are pulled towards the middle.
- **The worst 5% of houses make 21% of the error.** Most are big houses at low prices, and only the description says why: "needs some repairs", "second floor unfinished", "بيت شعبي" (an old traditional house). A few are odd ads: "for sale or for rent", or two duplexes for one price.
- **29 district names exist in more than one city**, used by 286 houses, a quarter of train. `district_norm` mixes الروضه in Riyadh with الروضه in Jeddah, so the target encoder learns one typical rent for two different places.

Also tried outside this notebook, with the same five parts, and dropped: words about the condition (repairs, old, unfinished), 23.9%, because only 1 to 14 houses mention each; seven more keyword flags (luxury, garden, sea and others), 23.5%; and a different size effect per city, 23.6%.

### Fix 1: keep districts apart by city

Join the city to the district name: `Riyadh|الروضه` and `Jeddah|الروضه` become two districts, each with its own typical rent. The new column goes on `df`, and `train` and `test` are made again from it. The split doesn't change.

In [ ]:
df["district_city"] = df["city_en"] + "|" + df["district_norm"]
train, test = split(df)

CATEGORICAL = ["city_en", "district_city", "front_en", "type"]
FEATURES = CATEGORICAL + NUMERIC
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
groups = train[FACTS].apply(house_text, axis=1)

print(train["district_norm"].nunique(), "district names became", train["district_city"].nunique(), "districts")

The two models again, with the new district column.

In [ ]:
ridge_inputs = ColumnTransformer([
    ("names", OneHotEncoder(handle_unknown="ignore"), NAMES),
    ("district", TargetEncoder(random_state=0), ["district_city"]),
    ("numbers", StandardScaler(), NUMERIC),
])
tree_inputs = ColumnTransformer(
    [
        ("names", OneHotEncoder(handle_unknown="ignore"), NAMES),
        ("district", TargetEncoder(random_state=0), ["district_city"]),
    ],
    remainder="passthrough",  # the numbers go through unchanged
)
ridge = on_log_scale(make_pipeline(ridge_inputs, Ridge(alpha=1)))
boosting = on_log_scale(make_pipeline(
    tree_inputs,
    HistGradientBoostingRegressor(learning_rate=0.05, max_leaf_nodes=15, max_iter=200),
))
results["Ridge, district within city"] = score(ridge, FEATURES)
results["Boosting, district within city"] = score(boosting, FEATURES)

### Fix 2: the average of both

The two models make different mistakes. Ridge sees straight-line effects, and the trees see combinations, so when one guesses too high and the other too low, the average lands closer.

For example, the real rent is 100,000. Ridge guesses 120,000 (+20%) and boosting 90,000 (−10%). Their average, taken on the log scale, is about 104,000 (+4%). `VotingRegressor` asks both models and averages their guesses.

In [ ]:
from sklearn.ensemble import VotingRegressor

both = on_log_scale(VotingRegressor([
    ("ridge", make_pipeline(ridge_inputs, Ridge(alpha=1))),
    ("boosting", make_pipeline(
        tree_inputs,
        HistGradientBoostingRegressor(learning_rate=0.05, max_leaf_nodes=15, max_iter=200),
    )),
]))
results["Average of both"] = score(both, FEATURES)

pd.DataFrame(results).T.sort_values("MAPE (%)")

### The choice

**The average of both, with districts kept apart by city**, has the lowest error: 21.6%, against 22.9% for Ridge with the same fix, and half the houses are guessed within 15%. That is more than 1 point better, so by the rule in `plan.md` the average wins. It needs both models: if the app ever runs it in a browser, the boosting part has to be converted first.

About 15 ideas were tried on the same five parts and the best was kept, so 21.6% is a little optimistic. Step 18 gives the honest number.

## Step 18: the final exam

The chosen model learns from all of `train` and guesses `test` once. These houses were set aside in step 3 and never used for any decision.

**Nothing changes after this cell.** Changing the model because of this score would turn `test` into one more practice exam, and the score would stop being honest.

B2, the baseline, takes the same exam for comparison only. It doesn't change the choice.

In [ ]:
both.fit(X_train, y_train)
test_guesses = pd.Series(both.predict(X_test), index=y_test.index)

b2.fit(X_train[["city_en", "log_size"]], y_train)
b2_guesses = pd.Series(b2.predict(X_test[["city_en", "log_size"]]), index=y_test.index)

pd.DataFrame({
    "Average of both": three_numbers(y_test, test_guesses),
    "B2: city + size": three_numbers(y_test, b2_guesses),
}).T

The same three numbers for each city.

In [ ]:
by_city = {}
for city, houses in test.groupby("city_en"):
    by_city[f"{city} ({len(houses)} houses)"] = three_numbers(
        y_test[houses.index], test_guesses[houses.index]
    )
pd.DataFrame(by_city).T

### The result

| On the 304 test houses | MAPE | MAE | within 15% |
| --- | --- | --- | --- |
| **The model: average of both** | **23.1%** | **22,871 SAR** | **46.4%** |
| B2: city + size | 38.8% | 34,562 SAR | 24.7% |

- **On houses it had never seen, the model's guess is off by 23.1% of the rent on average**, about 22,900 SAR, and almost half the houses are guessed within 15% of their real rent. The simple guess from city and size is off by 38.8%.
- **The cross-validation said 21.6%.** The exam is 1.5 points worse, as expected: that score was the best of about 15 tries, so it was a little lucky.
- **By city:** Jeddah 21.8% (87 houses) and Riyadh 24.7% (175 houses). Dammam (15.6%) and Al Khobar (22.8%) rest on only 23 and 19 houses, so their numbers are rough.
- **Where the score applies:** the cleaning rules removed test houses priced under 10,000 or over 500,000 SAR, because those prices aren't yearly rents. The score is for yearly rents between 10,000 and 500,000 SAR, and for the market of 2021, when the data was collected.

Nothing changes after this. Next: step 19, the model for the app.

## Step 19: the model for the app

The app shows a suggested rent beside Yearly rent on the Add and Edit property forms (FR-ADD-13), and a model makes it. It runs **Ridge**, not the average of both: Ridge is one weighted sum, so its numbers fit in a small file and a few lines of JavaScript, while the average would also need boosting's 200 trees in the browser.

That choice was made before Ridge took the exam, so its exam score below is honest, and it is the score the app's suggestions can claim.

In [ ]:
ridge.fit(X_train, y_train)
ridge_guesses = pd.Series(ridge.predict(X_test), index=y_test.index)
three_numbers(y_test, ridge_guesses)

Ridge scores as well as the average of both on the exam: 23.1%.

The exam is over, so the app's model learns from **every clean house**, train and test together. More houses give better typical rents per district. Its score stays the one above.

The app doesn't get a scikit-learn pipeline. It gets a list of numbers, and works out the log of the rent as:

base + a number for the city + one for the front + one for the type + one for the district + the sum of each of the 25 numbers × its weight

The rent is `exp` of that. The `StandardScaler` is folded into the numbers, so the app doesn't need it: each weight is divided by its column's scale, and the means move into the base. A district the model never saw gets `districtUnknown`, the value the target encoder gives an unknown district: the average over all houses.

`short` keeps 6 significant digits, which keeps the file small and changes no rent by more than 0.001%.

In [ ]:
import json

app_model = on_log_scale(make_pipeline(ridge_inputs, Ridge(alpha=1)))
app_model.fit(df[FEATURES], df[TARGET])

fitted = app_model.regressor_
inputs, weights = fitted[0], fitted[-1]
one_hot = inputs.named_transformers_["names"]
encoder = inputs.named_transformers_["district"]
scaler = inputs.named_transformers_["numbers"]

# The weights come in the order the ColumnTransformer builds its columns:
# one per one-hot category, then the district, then the 25 numbers.
coef = list(weights.coef_)
position = 0
name_weights = {}
for column, categories in zip(NAMES, one_hot.categories_):
    name_weights[column] = dict(zip(categories, coef[position:position + len(categories)]))
    position += len(categories)
district_weight = coef[position]
number_weights = coef[position + 1:]
base = weights.intercept_ - sum(w * m / s for w, m, s in zip(number_weights, scaler.mean_, scaler.scale_))


def short(value):
    """A number with 6 significant digits, to keep the file small."""
    return float(f"{value:.6g}")


rent_model = {
    "about": {
        "model": "Ridge regression on the log of the yearly rent, from ai-model/prepare.ipynb, step 19",
        "data": f"AQAR rental listings from 2021: {len(df)} houses after cleaning",
        "testMapePercent": float(three_numbers(y_test, ridge_guesses)["MAPE (%)"]),
    },
    # The sizes the model learned from. Outside them a weighted sum can give absurd rents,
    # so the app suggests nothing there.
    "sizeRange": {"min": int(df["size"].min()), "max": int(df["size"].max())},
    "base": short(base),
    "city": {name: short(weight) for name, weight in name_weights["city_en"].items()},
    "front": {name: short(weight) for name, weight in name_weights["front_en"].items()},
    "type": {name: short(weight) for name, weight in name_weights["type"].items()},
    "district": {
        name: short(district_weight * value)
        for name, value in zip(encoder.categories_[0], encoder.encodings_[0])
    },
    "districtUnknown": short(district_weight * encoder.target_mean_),
    "numbers": {column: short(w / s) for column, w, s in zip(NUMERIC, number_weights, scaler.scale_)},
}

with open("../frontend/src/data/rent-model.json", "w", encoding="utf-8", newline="\n") as file:
    json.dump(rent_model, file, ensure_ascii=False, indent=2)
    file.write("\n")
print(len(rent_model["district"]), "districts and", len(rent_model["numbers"]), "number weights written")

Check: the numbers in the file give the same rents as scikit-learn, for every clean house. `rent_from_file` works the rent out the way the app does. The `assert` stops with an error if any rent differs by 0.01% or more.

In [ ]:
def rent_from_file(row):
    """One house's rent, worked out from the file's numbers the way the app does it."""
    log_rent = rent_model["base"]
    log_rent += rent_model["city"].get(row["city_en"], 0)
    log_rent += rent_model["front"].get(row["front_en"], 0)
    log_rent += rent_model["type"].get(row["type"], 0)
    log_rent += rent_model["district"].get(row["district_city"], rent_model["districtUnknown"])
    for column, weight in rent_model["numbers"].items():
        log_rent += weight * row[column]
    return np.exp(log_rent)


from_file = df[FEATURES].apply(rent_from_file, axis=1)
from_sklearn = app_model.predict(df[FEATURES])
largest_gap = (from_file / from_sklearn - 1).abs().max()
print(f"largest gap: {largest_gap:.5%}")
assert largest_gap < 0.0001

Two made-up houses, not from the data, whose rents the app's tests check in `frontend/src/utils/rentModel.test.js`. If the model is exported again, copy the two new rents into that test.

- a new Riyadh villa in حي الملقا facing north: 400 m², 5 bedrooms, 5 bathrooms, 2 living rooms, a kitchen, a garage, a maid's room and air conditioning, described as "فيلا جديدة مع ملحق" (a new villa with an annex)
- a 10-year-old Jeddah duplex in a district the data doesn't have: 300 m², 4 bedrooms, 3 bathrooms, 1 living room, no front, no amenities and no description

In [ ]:
def made_up_house(city, district, front, kind, size, age, bedrooms, bathrooms, livingrooms, amenities, description):
    """One house's model inputs, computed the way steps 5 and 17 compute them."""
    text = normalise(description)
    house = {
        "city_en": city, "front_en": front, "type": kind,
        "district_city": city + "|" + district_name(district),
        "log_size": np.log10(size),
        "size_per_bedroom": size / max(bedrooms, 1),
        "property_age": age,
        "is_new": int(age == 0),
        "bedrooms": bedrooms, "bathrooms": bathrooms, "livingrooms": livingrooms,
        "total_rooms": bedrooms + bathrooms + livingrooms,
        "amenity_count": len(amenities),
        "is_corner": int(front in ["Three streets", "Four streets"]),
        "has_annex": int(bool(re.search(r"ملحق(?! به)", text))),
        "has_internal_stairs": int("درج داخلي" in text),
    }
    for amenity in AMENITIES:
        house[amenity] = int(amenity in amenities)
    return house


test_houses = pd.DataFrame([
    made_up_house("Riyadh", "حي الملقا", "North", "Villa", 400, 0, 5, 5, 2,
                  ["kitchen", "garage", "maid_room", "ac"], "فيلا جديدة مع ملحق"),
    made_up_house("Jeddah", "Unknown district", "", "Duplex", 300, 10, 4, 3, 1, [], ""),
])
app_model.predict(test_houses[FEATURES]).round(2)

How much worse is a guess when the app doesn't know the district, for example one the 2021 data doesn't have? The exam houses once more, with every district made unknown. This only measures the fallback: nothing about the model changes.

In [ ]:
unknown_districts = X_test.assign(district_city="unknown")
three_numbers(y_test, pd.Series(ridge.predict(unknown_districts), index=y_test.index))

### The district list for the form

The form's District field is a dropdown of the chosen city's districts. Its names come from the clean data: for each district, the spelling the ads use most, without the word حي in front. A district picked from this list always reaches the model's own number for it, so its suggestion is never the less precise "unknown district" one.

Each `assert` checks that a name, once normalised the way the app does it, gives back the model's district.

In [ ]:
spelling = (
    df["district"].str.strip()
    .str.replace(r"^حي\s+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)
most_used = (
    df.assign(name=spelling)
    .groupby(["city_en", "district_city"])["name"]
    .agg(lambda names: names.value_counts().index[0])
)

for (city, district), name in most_used.items():
    assert f"{city}|{district_name(name)}" == district, (district, name)
    assert district in rent_model["district"], district
most_used.groupby(level="city_en").size()

### English names for the dropdown

The app's labels are English, and the sample properties write their districts in English too (Al Malqa, Al Yasmin). So the dropdown shows each district's English name. The model knows a district only by its Arabic name, so `districts.json` keeps both: the English name that the form shows and saves, and the Arabic spelling from the cell above, which leads the app to the model's number.

The English names below are common spellings written out by hand, not taken from the data. To correct one, change it here and run the notebook again. The checks make sure that every district has an English name, that the table has no names left over, and that no two districts in a city share a name.

In [ ]:
ENGLISH_NAMES = {
    "Al Khobar": {
        "الامواج": "Al Amwaj",
        "الاندلس": "Al Andalus",
        "البحيرة": "Al Buhairah",
        "التحلية": "Al Tahliyah",
        "الثقبة": "Al Thuqbah",
        "الجسر": "Al Jisr",
        "الجوهرة": "Al Jawharah",
        "الحزام الاخضر": "Al Hizam Al Akhdar",
        "الحزام الذهبي": "Al Hizam Al Dhahabi",
        "الخبر الشمالية": "Al Khobar Al Shamaliyah",
        "الخزامى": "Al Khuzama",
        "الراكة الجنوبية": "Al Rakah Al Janubiyah",
        "السفن": "Al Sufun",
        "الشراع": "Al Shira",
        "الصواري": "Al Sawari",
        "العقربية": "Al Aqrabiyah",
        "العقيق": "Al Aqiq",
        "العليا": "Al Olaya",
        "الكورنيش": "Al Corniche",
        "اللؤلؤ": "Al Lulu",
        "المدينة الرياضية": "Sports City",
        "اليرموك": "Al Yarmuk",
        "قرطبة": "Qurtubah",
        "مدينة العمال": "Madinat Al Ummal",
    },
    "Dammam": {
        "الأمل": "Al Amal",
        "الاثير": "Al Athir",
        "الانوار": "Al Anwar",
        "الجوهرة": "Al Jawharah",
        "الحمراء": "Al Hamra",
        "الخالدية الشمالية": "Al Khalidiyah Al Shamaliyah",
        "الريان": "Al Rayyan",
        "السيف": "Al Saif",
        "الشاطئ الشرقي": "Al Shati Al Sharqi",
        "الشاطئ الغربي": "Al Shati Al Gharbi",
        "الشرق": "Al Sharq",
        "الشعلة": "Al Shulah",
        "الفرسان": "Al Fursan",
        "الفيصلية": "Al Faisaliyah",
        "المدينة الصناعية الثانية": "Second Industrial City",
        "المزروعية": "Al Mazruiyah",
        "المنار": "Al Manar",
        "المنتزه": "Al Muntazah",
        "النخيل": "Al Nakheel",
        "الندى": "Al Nada",
        "النزهة": "Al Nuzhah",
        "النورس": "Al Nawras",
        "بدر": "Badr",
        "جامعة الدمام": "Dammam University",
        "ضاحية الملك فهد": "King Fahd Suburb",
        "طيبة": "Taibah",
    },
    "Jeddah": {
        "ابحر الجنوبية": "Obhur Al Janubiyah",
        "ابحر الشمالية": "Obhur Al Shamaliyah",
        "الاجاويد": "Al Ajawid",
        "الاصالة": "Al Asalah",
        "الامواج": "Al Amwaj",
        "الامير فواز الشمالي": "Al Amir Fawaz Al Shamali",
        "الاندلس": "Al Andalus",
        "البساتين": "Al Basatin",
        "البغدادية الشرقية": "Al Baghdadiyah Al Sharqiyah",
        "البوادي": "Al Bawadi",
        "الجامعة": "Al Jamiah",
        "الحمدانية": "Al Hamdaniyah",
        "الحمراء": "Al Hamra",
        "الخالدية": "Al Khalidiyah",
        "الربوة": "Al Rabwah",
        "الرحاب": "Al Rehab",
        "الرحمانية": "Al Rahmaniyah",
        "الروضة": "Al Rawdah",
        "الرياض": "Al Riyadh",
        "الريان": "Al Rayyan",
        "الزمرد": "Al Zumurrud",
        "الزهراء": "Al Zahra",
        "السامر": "Al Samer",
        "السلامة": "Al Salamah",
        "الشاطئ": "Al Shati",
        "الشراع": "Al Shira",
        "الصالحية": "Al Salihiyah",
        "الصفا": "Al Safa",
        "الصوارى": "Al Sawari",
        "الفردوس": "Al Firdaws",
        "الفلاح": "Al Falah",
        "الفيصلية": "Al Faisaliyah",
        "الكوثر": "Al Kawthar",
        "اللؤلؤ": "Al Lulu",
        "المحمدية": "Al Muhammadiyah",
        "المرجان": "Al Murjan",
        "المروة": "Al Marwah",
        "المنتزه": "Al Muntazah",
        "النعيم": "Al Naeem",
        "النهضة": "Al Nahdah",
        "الورود": "Al Wurud",
        "الياقوت": "Al Yaqut",
        "ام حبلين": "Umm Hablain",
        "بريمان": "Briman",
        "بني مالك": "Bani Malik",
        "طيبة": "Taibah",
        "مريخ": "Muraykh",
        "مشرفة": "Mushrifah",
    },
    "Riyadh": {
        "أحد": "Uhud",
        "اشبيلية": "Ishbiliyah",
        "الازدهار": "Al Izdihar",
        "الاسكان": "Al Iskan",
        "الاندلس": "Al Andalus",
        "البديعة": "Al Badiah",
        "التعاون": "Al Taawun",
        "الجزيرة": "Al Jazirah",
        "الجنادرية": "Al Janadriyah",
        "الحزم": "Al Hazm",
        "الحمراء": "Al Hamra",
        "الخالدية": "Al Khalidiyah",
        "الخزامى": "Al Khuzama",
        "الخليج": "Al Khaleej",
        "الدار البيضاء": "Al Dar Al Baida",
        "الرائد": "Al Raid",
        "الربوة": "Al Rabwah",
        "الربيع": "Al Rabi",
        "الرحمانية": "Al Rahmaniyah",
        "الرفيعة": "Al Rafiah",
        "الرمال": "Al Rimal",
        "الروابي": "Al Rawabi",
        "الروضة": "Al Rawdah",
        "الزهرة": "Al Zahrah",
        "السعادة": "Al Saadah",
        "السفارات": "Diplomatic Quarter",
        "السلام": "Al Salam",
        "السليمانية": "Al Sulaimaniyah",
        "السويدي": "Al Suwaidi",
        "السويدي الغربي": "Al Suwaidi Al Gharbi",
        "الشرق": "Al Sharq",
        "الشفا": "Al Shifa",
        "الشهداء": "Al Shuhada",
        "الصحافة": "Al Sahafah",
        "الصفا": "Al Safa",
        "الضباط": "Al Dubbat",
        "العارض": "Al Arid",
        "العريجاء": "Al Uraija",
        "العريجاء الغربية": "Al Uraija Al Gharbiyah",
        "العريجاء الوسطى": "Al Uraija Al Wusta",
        "العزيزية": "Al Aziziyah",
        "العقيق": "Al Aqiq",
        "العليا": "Al Olaya",
        "الفلاح": "Al Falah",
        "الفيحاء": "Al Fayha",
        "القادسية": "Al Qadisiyah",
        "القدس": "Al Quds",
        "القيروان": "Al Qirawan",
        "المؤتمرات": "Al Mutamarat",
        "المحمدية": "Al Muhammadiyah",
        "المدينة الصناعية الجديدة": "New Industrial City",
        "المرسلات": "Al Mursalat",
        "المروج": "Al Muruj",
        "المصيف": "Al Masif",
        "المعيزلة": "Al Maizilah",
        "الملز": "Al Malaz",
        "الملقا": "Al Malqa",
        "الملك عبدالعزيز": "King Abdulaziz",
        "الملك عبدالله": "King Abdullah",
        "الملك فهد": "King Fahd",
        "الملك فيصل": "King Faisal",
        "المناخ": "Al Manakh",
        "المنار": "Al Manar",
        "المنصورة": "Al Mansurah",
        "المهدية": "Al Mahdiyah",
        "المونسية": "Al Munsiyah",
        "النخيل": "Al Nakheel",
        "الندى": "Al Nada",
        "النرجس": "Al Narjis",
        "النزهة": "Al Nuzhah",
        "النسيم الشرقي": "Al Naseem Al Sharqi",
        "النسيم الغربي": "Al Naseem Al Gharbi",
        "النفل": "Al Nafl",
        "النموذجية": "Al Namudhajiyah",
        "النهضة": "Al Nahdah",
        "الهدا": "Al Hada",
        "الواحة": "Al Wahah",
        "الوادي": "Al Wadi",
        "الورود": "Al Wurud",
        "الوزارات": "Al Wizarat",
        "الوشام": "Al Wisham",
        "الياسمين": "Al Yasmin",
        "اليرموك": "Al Yarmuk",
        "ام الحمام الشرقي": "Umm Al Hamam Al Sharqi",
        "ام الحمام الغربي": "Umm Al Hamam Al Gharbi",
        "ام سليم": "Umm Sulaym",
        "بدر": "Badr",
        "بنبان": "Banban",
        "جرير": "Jarir",
        "حطين": "Hittin",
        "ديراب": "Dirab",
        "سلطانة": "Sultanah",
        "شبرا": "Shubra",
        "طويق": "Tuwaiq",
        "ظهرة البديعة": "Dhahrat Al Badiah",
        "ظهرة لبن": "Dhahrat Laban",
        "عرقة": "Irqah",
        "عكاظ": "Okaz",
        "عليشة": "Ulaishah",
        "غبيرة": "Ghubairah",
        "غرناطة": "Ghirnatah",
        "قرطبة": "Qurtubah",
        "لبن": "Laban",
        "مطار الملك خالد الدولي": "King Khalid International Airport",
        "منفوحة الجديدة": "Manfuhah Al Jadidah",
    },
}

english = pd.Series(
    [ENGLISH_NAMES[city].get(name) for (city, _), name in most_used.items()],
    index=most_used.index,
)
assert english.notna().all(), most_used[english.isna()]
assert len(english) == sum(len(names) for names in ENGLISH_NAMES.values()), "names left over"
for city, names in english.groupby(level="city_en"):
    assert names.is_unique, names[names.duplicated()]

districts = {
    city: dict(sorted(zip(english[city], most_used[city])))
    for city in most_used.index.unique(level="city_en")
}

with open("../frontend/src/data/districts.json", "w", encoding="utf-8", newline="\n") as file:
    json.dump(districts, file, ensure_ascii=False, indent=2)
    file.write("\n")
{city: list(names)[:3] for city, names in districts.items()}

### Save the model as a file

The JSON above is for the app. For Python we also save the trained model itself, with **joblib**, scikit-learn's own save format. Later, `joblib.load("models/rent-model.joblib")` gives back the same model, and `.predict()` works at once, with no training.

- The file lands in `ai-model/models/`, which git ignores. It is not committed: only `rent-model.json` was approved (the dataset's licence).
- A joblib file runs code when it is loaded, so only load one you made yourself.
- It loads only with the same scikit-learn version, 1.7.2 (`requirements.txt`).

The cell loads the file back and checks it predicts the same rents as the model in memory.

In [ ]:
import joblib

os.makedirs("models", exist_ok=True)
joblib.dump(app_model, "models/rent-model.joblib")

again = joblib.load("models/rent-model.joblib")
assert np.allclose(again.predict(df[FEATURES]), app_model.predict(df[FEATURES]))
print(f"models/rent-model.joblib saved: {os.path.getsize('models/rent-model.joblib') / 1024:.0f} KB")

### What the app gets

- **`frontend/src/data/rent-model.json`**, about 7 KB: the base, 4 city, 10 front and 2 type numbers, 203 district numbers and 25 weights. It holds no rows of the dataset.
- **`ai-model/models/rent-model.joblib`**, for Python only: the same model saved with joblib. Git ignores it.
- **`frontend/src/data/districts.json`**: each city's districts, for the form's District dropdown: the English name it shows, and the Arabic spelling that leads the app to the model's own number for that district, so a district picked from the list is never unknown.
- **Ridge's honest score:** 23.1% average error on the 304 exam houses (22,998 SAR, and 46.1% of the houses within 15%).
- **An unknown district** falls back to the average district, and the error rises from 23.1% to 31.7%. So the app says when it doesn't recognise the district.
- **Only the sizes it learned from:** `sizeRange` keeps the smallest and largest clean house, 100 to 1,652 m². Outside that range a weighted sum gives absurd rents: for 90,000 m² it gives millions of riyals. So the app suggests nothing there.

Next: step 20 in `plan.md`, the write-up.